# declare

In [32]:
from pathlib import Path

import lab
from artifacts.core.resolve import resolve
from config import STORAGE

ROOT = STORAGE                              # the volume, inside the lab container
# ROOT = Path(".scratch/storage").resolve()  # a folder of your own, on a laptop

In [33]:
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import (
    LoopConfig,
    LRSchedule,
    OptimizerParameters,
    TrainingParameters,
)
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer

odyssey = SourceURL(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = SourceURL(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
romeojuliet = SourceURL(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
montecristo = SourceURL(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
pride = SourceURL(name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
frankenstein = SourceURL(name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt")
# greatexpectations = SourceURL(name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt")
# dracula = SourceURL(name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt")
# alice = SourceURL(name="alice", url="https://www.gutenberg.org/cache/epub/11/pg11.txt")
# sherlock = SourceURL(name="sherlock", url="https://www.gutenberg.org/cache/epub/1661/pg1661.txt")
# warandpeace = SourceURL(name="warandpeace", url="https://www.gutenberg.org/cache/epub/2600/pg2600.txt")
# taleoftwocities = SourceURL(name="taleoftwocities", url="https://www.gutenberg.org/cache/epub/98/pg98.txt")
# janeeyre = SourceURL(name="janeeyre", url="https://www.gutenberg.org/cache/epub/1260/pg1260.txt")
# huckfinn = SourceURL(name="huckfinn", url="https://www.gutenberg.org/cache/epub/76/pg76.txt")
# wutheringheights = SourceURL(name="wutheringheights", url="https://www.gutenberg.org/cache/epub/768/pg768.txt")

RUN_ID = "new_run"  # <- change this per run

tokenizer = BPETokenizer(
    vocab_size=800,
    special_tokens=("<pad>", "<unk>","<|endoftext|>"),
    sources=(mobydick, montecristo),
    allocated_resources=Resources(cpu=1)
)


mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(
        montecristo,
        mobydick,
        frankenstein,
    ),
    valid_sources=(romeojuliet, pride),
)


optimizer_parameters = OptimizerParameters(
    lr= 0.01,
    betas= (0.9, 0.999),
    weight_decay = 0.1,
    eps = 1e-8,
)

model = Transformer(
    vocab_size=tokenizer.vocab_size,
    sequence_length=64,
    num_layers=5,
    d_model=64,
    d_ff=128,
    num_heads=4,
    rope_theta=10000,
    device="cuda",  # has to agree with app.GPU; the preflight checks it
    dtype="torch.float32", #strings parsed in the worker container.
)

lr_schedule = LRSchedule(
    max_learning_rate=1e-3,
    min_learning_rate=1e-4,
    warmup_iters=100,
    cosine_cycle_iters=10000,
)

training_parameters = TrainingParameters(
    total_steps=1000,
    batch_size=64,
    max_norm=1.0,
    lr_schedule=lr_schedule,
    optimizer="torch.optim.AdamW",
    optimizer_parameters=optimizer_parameters,
    seed=0,
)

loop_config = LoopConfig(
    checkpoint_every=100,
    val_every=100,
    gpu_check_every=100,
    optimizer_checkpoint_policy="latest",
)

pretraining_0 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    model=model,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining_1 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    starting_checkpoint=pretraining_0,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining = pretraining_1

In [34]:
for artifact in resolve(pretraining):
    print(f"{type(artifact).__name__:16} {artifact.artifact_path}")

SourceURL        sources/montecristo
SourceURL        sources/mobydick
Tokenizer        tokenizers/bpe-800-f90570d566
TokenizedSource  tokenized/bpe-800-f90570d566/montecristo
TokenizedSource  tokenized/bpe-800-f90570d566/mobydick
SourceURL        sources/frankenstein
TokenizedSource  tokenized/bpe-800-f90570d566/frankenstein
SourceURL        sources/romeojuliet
TokenizedSource  tokenized/bpe-800-f90570d566/romeojuliet
SourceURL        sources/pride
TokenizedSource  tokenized/bpe-800-f90570d566/pride
MappedDataSet    mappeddatasets/mapped-437fb6c55f
Pretraining      runs/new_run/pretraining/0-1000-06e8fbe39f
Pretraining      runs/new_run/pretraining/1000-2000-b62d0c9474


In [36]:
from local import declare_on_volume
# report = lab.declare(pretraining, root=ROOT, verbose=True, commit=True) #local
report = declare_on_volume(pretraining, verbose=True, commit=True)  #cloud

sources/montecristo                            declared  (created)
sources/mobydick                               declared  (created)
tokenizers/bpe-800-f90570d566                  declared  (created)
tokenized/bpe-800-f90570d566/montecristo       declared  (created)
tokenized/bpe-800-f90570d566/mobydick          declared  (created)
sources/frankenstein                           declared  (created)
tokenized/bpe-800-f90570d566/frankenstein      declared  (created)
sources/romeojuliet                            declared  (created)
tokenized/bpe-800-f90570d566/romeojuliet       declared  (created)
sources/pride                                  declared  (created)
tokenized/bpe-800-f90570d566/pride             declared  (created)
mappeddatasets/mapped-437fb6c55f               declared  (created)
runs/new_run/pretraining/0-1000-06e8fbe39f     declared  (created)
runs/new_run/pretraining/1000-2000-b62d0c9474  declared  (created)

14 declared
14 manifests written: sources/montecristo, source

In [ ]:
# report = lab.declare(pretraining, root=ROOT, commit=True)

## Declare on the volume, from here

The volume's mount exists only inside a container, so declaring onto it from
a laptop is one explicit remote call. `local.declare_on_volume` has
`lab.declare`'s form minus `root`: same report printed, same report
returned, same refusal on a blocker. It runs an ephemeral Modal app per call.


## Running by hand

The launcher runs jobs on Modal from the dashboard. To build something small
right here instead, walk its resolution and run each producer that has not
written its files yet. `lab.worker` stands in for the execution context a
real call gets.

In [ ]:
for artifact in resolve(justdataset):
    if artifact.producer is None or all(artifact.status(ROOT).completion.values()):
        continue
    artifact.job().run(ROOT, lab.worker)

In [ ]:
dset = justdataset.bind(ROOT)

In [ ]:
dset.valid_tokens

## Check a finished leg

Once a leg's `model.pt` is on the volume, `bind(ROOT)` builds its model and
loads the saved weights onto it. One validation batch through it is the
quickest confirmation that what trained is what was declared. The leg
declares `device="cuda"`; binding in the lab, which has no GPU, loads on the
CPU instead, so the batch goes wherever the model landed.

In [ ]:
from artifacts.stages.pretraining.jobs import get_batch

leg = pretraining.bind(ROOT)
inputs, labels = get_batch(
    leg.dataset.bind(ROOT).valid_tokens,
    leg.training_parameters.batch_size,
    leg.model.sequence_length,
    leg.training_parameters.seed,
    0,  # step 0's batch, the one its first training step drew
    next(leg.live_model.parameters()).device,
)
leg.live_model(inputs).shape  # (batch_size, sequence_length, vocab_size)

In [16]:
tokenizer = BPETokenizer(
    vocab_size=800,
    special_tokens=("<pad>", "<unk>","<|endoftext|>"),
    sources=(mobydick, montecristo),
    allocated_resources=Resources(cpu=1)
)


In [19]:
tokenizer.

TypeError: Artifact.load() missing 1 required positional argument: 'artifact_path'

In [ ]:
artifact